In [ ]:
import functools as ft
import itertools
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.tree as jt
import matplotlib.pyplot as plt
import numpy as np
import optax
import seaborn as sns
from jaxtyping import Array, Float, Integer, Key, Scalar, ScalarLike
from mdpax.core.problem import Problem
from mdpax.utils.spaces import create_range_space
from mdpax.utils.types import (
    ActionSpace,
    ActionVector,
    Policy,
    RandomEventSpace,
    RandomEventVector,
    Reward,
    StateSpace,
    StateVector,
)
from tqdm.notebook import tqdm

import fairsim
from fairsim.learning.net import (
    DecisionInput,
    DecisionNet,
    DecisionUnnet,
    fair_loss,
    policy_and_loss,
)
from fairsim.model.joint import (
    joint_opt_demographic_parity,
    joint_opt_equal_opportunity,
    ternary_maximize_1d,
)
from fairsim.model.mdp import (
    Setting,
    construct_transition,
    transition_kernel_construct,
)
from fairsim.model.myopic import (
    opt_choice_prob,
    opt_tpr,
    opt_unconstrained,
)
from fairsim.util import KeyGen, tree_stack, tree_unstack

sns.set_theme(context="notebook", style="whitegrid")

In [ ]:
nets: dict[str, DecisionNet | DecisionUnnet] = {}

discount = 0.6

def noise_kernel(k):
    return jax.scipy.stats.binom.pmf(jnp.arange(0, k*2+1), n=k*2, p=0.5)

n_scores = 17
kernel = transition_kernel_construct(
    repay=jnp.array([0, 80, 20]),
    default=jnp.array([100, 0, 0]),
    reject=jnp.array([10, 70, 20]),
    noise=jnp.array([1]),
)

# n_scores = 21
# kernel = construct_transition(
#     repay=jnp.array([0, 0, 0, 0, 5, 4, 0, 0, 0]),
#     default=jnp.array([6, 6, 0, 0, 0, 0, 0, 0, 0]),
#     reject=jnp.array([0, 0, 0, 0, 1, .25, 0, 0, 0]),
#     # noise=jnp.array([1, 4, 6, 4, 1]),
#     # noise=jnp.array([1]),
#     noise=noise_kernel(6),
# )

succ_prob = jnp.linspace(0.2, 0.8, n_scores)
# succ_prob = jnp.sin(jnp.linspace(-1, 1, n_scores) * jnp.pi / 2)**3 * 0.3 + 0.5
reward = succ_prob - .7
setting = Setting(
    reward=reward,
    trans=kernel,
    success_prob=succ_prob,
)

weights = jnp.array([0.2, 0.8])
weights = weights / weights.sum()
n_groups = len(weights)


In [ ]:
learning_rate = 3e-3
batch_size = 4096
discount = 0.99
n_iterations = 768
n_steps = 12000
schedule = optax.schedules.exponential_decay(
    init_value=learning_rate,
    transition_begin=500,
    transition_steps=4000,
    decay_rate=0.1,
    end_value=1e-6,
)
kg = KeyGen(2025)
checkpoint_path = Path("./checkpoints/")
checkpoint_path.mkdir(exist_ok=True)

# for constraint_mode in ["eo", "dp"]:
for constraint_mode in ["unconstrained", "eo", "dp"]:
    model_key = kg()

    net = DecisionNet(
        n_states=n_scores,
        n_groups=n_groups,
        n_hidden=4,
        w_hidden=n_scores * n_groups * 8,
        key=model_key,
    )

    opt = optax.adam(learning_rate)
    opt_state = opt.init(net)  # type: ignore


    def generate_distribution(
        *,
        key: Key[Scalar, ""],
    ) -> DecisionInput:
        def generate_single(key):
            return jr.dirichlet(key, alpha=jnp.ones(n_scores))

        return jax.vmap(generate_single)(jr.split(key, n_groups))


    @jax.jit
    def step(
        model: DecisionNet, opt_state: optax.OptState, *, key: Key[Scalar, ""]
    ):
        def batch_loss(model):
            def single_loss(key):
                dist = generate_distribution(key=key)

                def iter(dist, _):
                    policy, loss, _ = policy_and_loss(
                        model,
                        dist,
                        weights,
                        setting,
                        constraint_type=constraint_mode,
                    )
                    dist = jax.vmap(
                        setting.transition
                    )(dist, policy)
                    return dist, loss

                final_dist, losses = jax.lax.scan(
                    iter, dist, None, length=n_iterations
                )
                return losses * (discount ** jnp.arange(n_iterations)).sum()

            keys = jr.split(key, batch_size)
            losses = jax.vmap(single_loss)(keys)
            return losses.mean()

        loss_value, grads = jax.value_and_grad(batch_loss)(model)
        updates, opt_state = opt.update(grads, opt_state)
        model = optax.apply_updates(model, updates)  # type: ignore
        return model, opt_state, loss_value
    progress = tqdm(range(n_steps))
    loss_mean = 0.0
    loss_mean_denom = 0.0
    loss_horizon = 50
    for step_idx in progress:
        net, opt_state, loss_value = step(
            net,
            opt_state,
            key=kg(),
        )
        loss_mean = loss_mean * (loss_horizon - 1) / loss_horizon + loss_value / loss_horizon
        loss_mean_denom = loss_mean_denom * (loss_horizon - 1) / loss_horizon + 1 / loss_horizon
        nets[f"nn-{constraint_mode}"] = net
        progress.set_description(f"mode: {constraint_mode}, loss: {loss_mean / loss_mean_denom:.4f}")

In [ ]:
for name, net in nets.items():
    eqx.tree_serialise_leaves(
        checkpoint_path / f"{name}.eqx",
        net,
    )

In [ ]:
gpu_device = jax.devices('gpu')[0]
cpu_device = jax.devices('cpu')[0]

In [ ]:
n_iter = 10000
distrib_x = 1 + jnp.arange(n_scores)
distrib_x = distrib_x / distrib_x.sum()
init_distribs = jnp.stack([distrib_x[::-1], distrib_x])


@ft.partial(jax.jit, device=cpu_device, static_argnames=["mode"])
def sim_step(distribs, _, mode):
    if mode == "dp":
        th, policies = joint_opt_demographic_parity(distribs, weights, reward)
    elif mode == "eo":
        th, policies = joint_opt_equal_opportunity(
            distribs, weights, succ_prob, reward
        )
    elif mode == "unconstrained":
        policies = jax.vmap(ft.partial(opt_unconstrained, rew=reward))(distribs)
        th = None
    elif mode.startswith("nn"):
        net = nets[mode]
        policies = policy_and_loss(
            net,
            distribs,
            weights,
            setting,
            constraint_type=mode[3:],
        )[0]
        th = None
    else:
        assert False, f"Unknown mode {mode}"
    if th is None:
        th = jnp.einsum("i,ij,ij->", weights, distribs, policies)
    distribs_ = jax.vmap(setting.transition)(distribs, policies)
    distribs_ = jax.vmap(lambda x: x / x.sum())(distribs_)
    return distribs_, (distribs, policies, th)


def dist(distribs):
    cdfs = jax.vmap(jnp.cumsum)(distribs)
    return cdfs.ptp(axis=0).max()


result = {}
for mode in tqdm(["dp", "eo", "unconstrained", "nn-dp", "nn-eo", "nn-unconstrained"]):
    if mode.startswith("nn"):
        if mode not in nets:
            continue
    distribs, (history, policies, ths) = jax.jit(
        lambda z: jax.lax.scan(
            ft.partial(sim_step, mode=mode), z, None, length=n_iter + 1
        )
    )(init_distribs)
    dists = jax.vmap(dist)(history)
    result[mode] = {
        "history": history,
        "thresholds": ths,
        "policies": policies,
        "distances": dists,
    }

In [ ]:
def is_threshold(pol, eps: float=1e-3):
    vals = jnp.minimum(pol, 1-pol)
    return jnp.sum(vals > eps) <= 1

In [ ]:
sampled_times = [0, 4, 16, 64, 256, 1024]
fig, ax = plt.subplots(
    len(result.keys()),
    len(sampled_times),
    figsize=(3 * len(sampled_times), 1.5 * len(result.keys()) + 3),
    dpi=200,
    layout="constrained",
    sharex=True,
)
ht = {mode: tree_unstack(result[mode]) for mode in result.keys()}
for ax, (mode, t) in zip(
    ax.flatten(), itertools.product(result.keys(), sampled_times)
):
    # ax_ = ax.twinx()
    for distrib, policy in zip(ht[mode][t]["history"], ht[mode][t]["policies"]):
        # sns.barplot(x=jnp.arange(n_scores), y=distrib, ax=ax, alpha=0.5)
        bar = ax.bar(
            jnp.arange(n_scores) + 0.5, distrib, alpha=0.5, linewidth=0.0, width=1.0
        )
        color = bar.patches[0].get_facecolor()
        # if is_threshold(policy):
        if not mode.startswith("nn") or mode == "nn-unconstrained":
            vline_pos = (1 - policy).sum()
            ax.axvline(vline_pos, color=color[:3], linestyle="--")
        # ax_.plot(
        #     jnp.arange(n_scores) + 0.5,
        #     policy,
        #     color=color[:3],
        # )
        # sns.lineplot(x=jnp.arange(n_scores), y=distrib.cumsum(), ax=ax)
    ax.set_title(f"{names[mode]}, t={t}")
    # change the y tick labels to percentage
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
fig.get_layout_engine().set(w_pad=4 / 72, h_pad=4 / 72, hspace=0.05,
                            wspace=0.051)
plt.show()